### Importando as funções e bibliotecas

In [ ]:
import os
import sys

project_root = os.path.abspath(os.path.join(os.getcwd(), '..'))
if project_root not in sys.path:
    sys.path.append(project_root)

from src.data_processing import process_data
from src.model_training import train_ridge_regression, train_random_forest, train_lgbm
from src.evaluation import evaluate_model_performance, save_results, save_model
from sklearn.model_selection import train_test_split
import pandas as pd

### Criando automaticamente um DataFrame processado e pronto para modelagem

In [ ]:
input_path = '../data/raw/birds.csv'
output_path = '../data/processed/proc_birds.csv'

df = process_data(input_path=input_path, output_path=output_path)
df.head(3)

Iniciando o processamento do meta-dataset...
Dataset carregado com 135915 linhas e 209 colunas

Removendo linhas NaN...
Removidas 31171 linhas. Restaram 104744 linhas validas

Processando todas as colunas...

Processamento concluido!:
total de NaNs após o processamento: 0

Dataset salvo com sucesso em: ../data/processed/proc_birds.csv


,normalize,feature preprocessing,mlfs.igmf_adapted.PyIT_IGMF,mlfs.igmf_adapted.PyIT_IGMF-n_features,mlfs.ppt_mi_adapted.PyIT_PPT_MI,mlfs.ppt_mi_adapted.PyIT_PPT_MI-n_features,mlfs.scls_adapted.PyIT_SCLS,mlfs.scls_adapted.PyIT_SCLS-n_features,mlfs.lrfs_adapted.PyIT_LRFS,mlfs.lrfs_adapted.PyIT_LRFS-n_features,...,weka.classifiers.functions.supportVector.NormalizedPolyKernel-E,weka.classifiers.functions.supportVector.NormalizedPolyKernel-L,weka.classifiers.functions.supportVector.Puk,weka.classifiers.functions.supportVector.Puk-O,weka.classifiers.functions.supportVector.Puk-S,weka.classifiers.functions.supportVector.RBFKernel,weka.classifiers.functions.supportVector.RBFKernel-G,F1 (macro averaged by label),Model Size,Model Size Log
0,0,1,0,-1.0,0,-1.0,0,-1.0,0,-1.0,...,-1.0,-1.0,0,-1.0,-1.0,0,-1.0,0.256,37076.0,10.520752
1,0,1,0,-1.0,0,-1.0,0,-1.0,0,-1.0,...,-1.0,-1.0,0,-1.0,-1.0,0,-1.0,0.222,29686.0,10.298465
2,0,1,0,-1.0,0,-1.0,0,-1.0,0,-1.0,...,-1.0,-1.0,0,-1.0,-1.0,0,-1.0,0.026,18387.0,9.819454


### Dividindo o dataset processado em treino e teste
Essa parte deve ser mantida aqui para ser reutilizável em modelos diferentes 

In [2]:
# Essa parte é mantida no notebook para reutilização em todas os modelos
df = pd.read_csv('../data/processed/proc_birds.csv')

# Separar features e targets
X = df.drop(columns=['F1 (macro averaged by label)', 'Model Size', 'Model Size Log'])
y = df[['F1 (macro averaged by label)', 'Model Size Log']]

# Dividir em treino e teste 
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

### Treinando modelos com MultiOutputRegressor
Estratégia de separar o problema de multitarget em targets únicos e treinar o modelo para prever cada um separadamente

#### Treinando o modelo Ridge Regression

In [ ]:
# Chamar a função que treina e otimiza o modelo
grid_search_results = train_ridge_regression(
    X_train=X_train, 
    y_train=y_train, 
    alpha_values=[7.0, 8.0, 9.0]
)

# O best_estimator_ é o modelo já treinado com o melhor hiperparâmetro 
model = grid_search_results.best_estimator_

# Fazer predições
predictions = model.predict(X_test)

# Obter e exibir as métricas 
final_metrics = evaluate_model_performance(y_test, predictions)
print("Relatório:")
for metric_name, score in final_metrics.items():
    print(f"{metric_name}: {score}")

# Salvar os resultados deste modelo
save_results(
    model_name='Ridge Regression',
    metrics=final_metrics,
    best_params=grid_search_results.best_params_
)

# Salvar o modelo para que seja reutilizável
save_model(model, '../models/ridge.joblib')

Iniciando o GridSearch...
Fitting 5 folds for each of 3 candidates, totalling 15 fits
Melhores parâmetros encontrados: {'multioutputregressor__estimator__alpha': 7.0}
Melhor R² (média do cross validation): 0.8478

Relatório:
R2 para F1 (macro averaged by label): 0.7511
R2 para Model Size Log: 0.9431
RMSE para F1 (macro averaged by label): 0.0738
RMSE para Model Size Log: 0.5115
Resultados para o modelo 'Ridge Regression' salvos com sucesso em ../reports/model_comparison.csv


#### Treinando o modelo RandomForest

In [ ]:
# Chamar a função
# A variável optuna_study não é usada diretamente, mas fica disponível caso necessário
best_model, best_params, optuna_study = train_random_forest(
    X_train=X_train, 
    y_train=y_train,
    n_trials=50  
)

# 'best_model' é o pipeline treinado com os melhores hiperparâmetros
predictions = best_model.predict(X_test)

# Obter e exibir as métricas 
final_metrics = evaluate_model_performance(y_test, predictions)

print("\nRelatório:")
for metric_name, score in final_metrics.items():
    print(f"{metric_name}: {score}")

# Salvar os resultados deste modelo
save_results(
    model_name='RandomForest',
    metrics=final_metrics,
    best_params=best_params,
    filepath ='../reports/model_comparison_birds.csv'
)

# Salvar o modelo 
save_model(
    model=best_model, 
    filepath='../models/multi_output/random_forest.joblib'
)

Iniciando o RandomSearch...
Fitting 2 folds for each of 1 candidates, totalling 2 fits
Melhores parâmetros encontrados: {'multioutputregressor__estimator__n_estimators': 500, 'multioutputregressor__estimator__min_samples_split': 5, 'multioutputregressor__estimator__min_samples_leaf': 1, 'multioutputregressor__estimator__max_features': 'sqrt', 'multioutputregressor__estimator__max_depth': 40, 'multioutputregressor__estimator__bootstrap': False}
Melhor R² (média do cross validation): 0.9390

Relatório:
R2 para F1 (macro averaged by label): 0.9108
R2 para Model Size Log: 0.9872
RMSE para F1 (macro averaged by label): 0.0442
RMSE para Model Size Log: 0.2427
Resultados para o modelo 'RandomForest Regression' salvos com sucesso em ../reports/model_comparison.csv


#### Treinando o modelo LightGBM

In [ ]:
# Chamar a função que treina e otimiza o modelo
lgbm_search_results = train_lgbm(
    X_train=X_train, 
    y_train=y_train, 
)

# O best_estimator_ é o modelo já treinado com o melhor hiperparâmetro 
model = lgbm_search_results.best_estimator_

# Fazer predições
predictions = model.predict(X_test)

# Obter e exibir as métricas 
final_metrics = evaluate_model_performance(y_test, predictions)
print("Relatório:")
for metric_name, score in final_metrics.items():
    print(f"{metric_name}: {score}")

# Salvar os resultados deste modelo
save_results(
    model_name='LGBM Regression',
    metrics=final_metrics,
    best_params=lgbm_search_results.best_params_
)

# Salvar o modelo para que seja reutilizável
save_model(model, '../models/lgbm.joblib')